# Aula 3: filtrar e resumir

Na aula 2 arrumamos os tipos de uma base. Agora vamos usar esses tipos para
duas coisas: **recortar** o que entra na análise e **resumir** a coluna em um
número.

O notebook segue o mesmo formato: cada operação aparece primeiro resolvida, e
depois vem um bloco **Agora você** com a mesma operação em outra coluna.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


## A pergunta de pesquisa

> Nos acórdãos do TJSP que arbitram indenização por dano moral, qual é o valor
> típico, e quanto ele varia?

"Valor típico" e "quanto varia" são as duas perguntas que a estatística
descritiva responde: uma medida de **posição** e uma medida de **dispersão**.
Escolher qual é o conteúdo da aula.


### De onde vieram os dados

```python
import juscraper as jus

tjsp = jus.scraper("tjsp")
acordaos = tjsp.cjsg('"dano moral" E "arbitro a indenizacao"', paginas=range(1, 26))
```


In [ ]:
danos = pd.read_csv(f"{URL}/tjsp_cjsg_dano_moral.csv")
danos.head(3)


In [ ]:
danos.info()


### As colunas que vieram do texto

Quatro colunas desta base não vieram prontas do tribunal: elas foram **lidas do
texto da ementa** antes de o arquivo ser publicado.

| coluna | como foi definida |
|---|---|
| `valor_indenizacao` | o primeiro valor em reais que aparece na ementa |
| `tem_dano_moral` | a ementa menciona a expressão "dano moral" |
| `houve_majoracao` | a ementa menciona alguma forma de "majorar" |
| `camara` e `secao` | o número e a seção lidos do nome do órgão julgador |

A ferramenta que faz essa leitura é a expressão regular, e ela é o assunto da
aula 12. Aqui interessa outra coisa: **as decisões que essas definições
embutem**, e o que elas custam.

Veja uma ementa e a linha correspondente da tabela:


In [ ]:
print(danos.loc[0, "ementa"][:700])


In [ ]:
danos.loc[[0], ["processo", "valor_indenizacao", "tem_dano_moral", "houve_majoracao"]]


Duas limitações que precisam estar escritas em qualquer relatório que use estas
colunas:

- `valor_indenizacao` pega o **primeiro** valor em reais da ementa. Nem sempre é
  o valor arbitrado: pode ser o valor pedido, o da sentença reformada, ou custas;
- `tem_dano_moral` marca a ementa que **menciona** dano moral, inclusive para
  dizer que não é caso de indenizar.

Nenhuma das duas invalida o exercício. As duas mudam o que se pode concluir.


**Exercício 1.** Escolha uma variável que você gostaria de ter nesta tabela e
que não está lá. Escreva o nome dela, o tipo, e a instrução que faria duas
pessoas lerem a mesma ementa e registrarem o mesmo valor. Não precisa programar.
Este é exatamente o trabalho que o Projeto 1 vai pedir hoje.


In [ ]:
# exercício 1 (responda em célula de texto)


### Retomando: que tipo é cada coluna

Duas já estão preenchidas como modelo.


In [ ]:
tipos = {
    # modelo
    "processo": "identificador",
    "ementa": "texto",
    # complete
    "comarca": "________",
    "camara": "________",
    "secao": "________",
    "data_julgamento": "________",
    "valor_indenizacao": "________",
    "tem_dano_moral": "________",
    "n_palavras_ementa": "________",
}

pd.Series(tipos).value_counts()


`camara` merece atenção: ela é escrita com dígitos e mesmo assim é categórica
nominal. A 13ª câmara não é maior que a 2ª.


### Convertendo as datas

Mesma operação da aula 2, com uma diferença: aqui a data vem no formato
brasileiro, `07/08/2026`, e o pandas precisa que você diga isso com `format=`.
`%d` é o dia, `%m` o mês e `%Y` o ano com quatro dígitos.


In [ ]:
danos["data_julgamento"] = pd.to_datetime(danos["data_julgamento"], format="%d/%m/%Y")

danos["data_julgamento"].head(3)


**Agora você.** Converta `data_publicacao`, que vem no mesmo formato.


In [ ]:
danos["data_publicacao"] = pd.to_datetime(danos["data_publicacao"], format="________")

danos[["data_julgamento", "data_publicacao"]].dtypes


## Variável, valor observado e estatística

Três palavras que a conversa do dia a dia mistura:

- **variável**: o que se mede, "valor da indenização em reais". Existe antes dos
  dados e é a coluna da tabela;
- **valor observado**: o valor de um caso, "neste acórdão foram R$ 5.000,00". É
  uma célula;
- **estatística**: um resumo de muitos valores observados, "a mediana foi
  R$ 5.000,00". É um número calculado da coluna inteira.

O tipo é propriedade da variável, e é ele que decide qual estatística faz
sentido.


## Filtrar

Recortar a base é decisão de pesquisa, não detalhe técnico. Há três jeitos de
fazer, e eles dão o mesmo resultado.

### 1. Índice lógico

Uma comparação devolve uma série de `True` e `False`, do mesmo tamanho do
DataFrame. Isso é uma máscara:


In [ ]:
tem_valor = danos["valor_indenizacao"].notna()

tem_valor.head()


E o DataFrame indexado pela máscara devolve só as linhas verdadeiras:


In [ ]:
danos[tem_valor].shape


**Agora você.** Crie a máscara `foi_majorado` a partir de `houve_majoracao`, que já é uma coluna de `True` e `False`, e veja quantas linhas sobram.


In [ ]:
foi_majorado = danos["________"]

danos[________].shape


Para combinar condições: `&` é "e", `|` é "ou", `~` é "não". Os parênteses em
volta de cada condição são obrigatórios.


In [ ]:
privado_com_valor = danos[(danos["secao"] == "Direito Privado") & tem_valor]

privado_com_valor.shape


**Agora você.** Monte um recorte com os acórdãos que têm valor **e** em que houve majoração.


In [ ]:
com_valor_e_majoracao = danos[tem_valor ________ foi_majorado]

com_valor_e_majoracao.shape


O índice lógico é o jeito mais explícito de filtrar, e é o único que vamos usar
hoje. Na aula 4 aparecem outros dois, `.loc` e `.query`, que escrevem a mesma
coisa de forma mais curta.

### O recorte da pergunta

Um terço das ementas não trouxe valor. A decisão aqui é analisar só quem tem
valor, e a consequência é que a resposta vale para **os acórdãos que trazem o
valor na ementa**, não para todos.


In [ ]:
com_valor = danos.dropna(subset=["valor_indenizacao"]).copy()

len(danos), len(com_valor)


## Posição: onde fica o centro

`.describe()` já dá vários resumos de uma vez:


In [ ]:
com_valor["valor_indenizacao"].describe().round(2)


As duas medidas de posição mais usadas são a média e a mediana. `.mean()`
devolve a média:


In [ ]:
com_valor["valor_indenizacao"].mean().round(2)


**Agora você.** Calcule a mediana, com `.median()`.


In [ ]:
com_valor["valor_indenizacao"].________()


A média é bem maior que a mediana. Isso é a assinatura de uma distribuição
**assimétrica à direita**: poucos valores muito altos puxam a média, e a mediana
ignora. Em valores monetários no Direito, isso é a regra, não a exceção.

Dá para ver o efeito tirando um único caso, com o mesmo índice lógico de antes:
mantemos só as linhas cujo valor é menor que o máximo.


In [ ]:
maior = com_valor["valor_indenizacao"].max()
sem_o_maior = com_valor[com_valor["valor_indenizacao"] < maior]

pd.DataFrame({
    "com todos": [
        com_valor["valor_indenizacao"].mean(),
        com_valor["valor_indenizacao"].median(),
    ],
    "sem o maior": [
        sem_o_maior["valor_indenizacao"].mean(),
        sem_o_maior["valor_indenizacao"].median(),
    ],
}, index=["média", "mediana"]).round(2)


Uma linha muda a média e não move a mediana. É por isso que "o valor típico da
indenização" quase sempre deve ser reportado com a mediana.


## Quantis: a distribuição inteira

O quantil de ordem $p$ é o valor abaixo do qual está a fração $p$ dos casos. A
mediana é o quantil 0,5. `.quantile()` aceita uma lista:


In [ ]:
com_valor["valor_indenizacao"].quantile([0.25, 0.50, 0.75]).round(2)


**Agora você.** Peça os quantis 0,10, 0,90 e 0,99, para enxergar as duas pontas.


In [ ]:
com_valor["valor_indenizacao"].quantile([________]).round(2)


Compare o quantil 0,90 com o máximo: a distância entre os dois é o tamanho da
cauda.

O intervalo interquartílico (IQR) é a largura da metade central dos dados, ou
seja, a distância entre o quantil 0,25 e o 0,75:


In [ ]:
q1 = com_valor["valor_indenizacao"].quantile(0.25)
q3 = com_valor["valor_indenizacao"].quantile(0.75)

pd.Series({
    "Q1": q1,
    "Q3": q3,
    "IQR": q3 - q1,
    "amplitude": com_valor["valor_indenizacao"].max() - com_valor["valor_indenizacao"].min(),
})


A amplitude é decidida por dois casos extremos. O IQR não, e por isso ele é a
medida de dispersão que acompanha a mediana.


## Dispersão: variância e desvio padrão

O desvio padrão mede o afastamento típico em relação à média. A conta divide por
$n - 1$, e não por $n$: é o desvio padrão **amostral**, que é o padrão do pandas.
O parâmetro que controla isso é o `ddof`, e ele vale 1 por omissão.


In [ ]:
com_valor["valor_indenizacao"].std().round(2)


**Agora você.** Calcule o desvio padrão populacional, passando `ddof=0`, e compare com o de cima.


In [ ]:
com_valor["valor_indenizacao"].std(ddof=________).round(2)


Refazendo a conta na mão, para ver que não tem mágica:


In [ ]:
x = com_valor["valor_indenizacao"]
n = len(x)

variancia = ((x - x.mean()) ** 2).sum() / (n - 1)

pd.Series({
    "variância na mão": variancia,
    "variância pandas": x.var(),
    "desvio padrão na mão": variancia ** 0.5,
    "desvio padrão pandas": x.std(),
}).round(2)


Com $n$ na casa das centenas, a diferença entre dividir por $n$ e por $n-1$ é
pequena. Com $n = 12$, que é o tamanho de muita amostra de pesquisa em Direito,
deixa de ser.

O desvio padrão sozinho é difícil de interpretar, porque ele vem na unidade da
variável. O **coeficiente de variação** põe a dispersão em escala relativa,
dividindo pelo valor médio:


In [ ]:
com_valor["valor_indenizacao"].std() / com_valor["valor_indenizacao"].mean()


Um coeficiente maior que 1 quer dizer que o desvio padrão é maior que a média:
os valores estão espalhadíssimos. Reportar só "a indenização média foi
R$ 7.935" sem dizer isso é enganoso.


## A média de uma binária é uma proporção

Este é o truque que mais aparece daqui em diante. Uma variável binária guardada
como `True` e `False` é lida pelo Python como 1 e 0. Somar dá o número de casos
`True`, e dividir pelo total dá a proporção. Ou seja: **a média é a proporção**.

Primeiro criamos a binária, com uma comparação:


In [ ]:
com_valor["acima_de_10k"] = com_valor["valor_indenizacao"] >= 10000

com_valor["acima_de_10k"].head()


In [ ]:
pd.Series({
    "quantos True": com_valor["acima_de_10k"].sum(),
    "total": len(com_valor),
    "soma dividida pelo total": com_valor["acima_de_10k"].sum() / len(com_valor),
    "média": com_valor["acima_de_10k"].mean(),
}).round(4)


**Agora você.** Calcule a proporção de acórdãos em que houve majoração, usando `.mean()` na coluna `houve_majoracao`.


In [ ]:
com_valor["houve_majoracao"].________().round(4)


O desvio padrão de uma binária também não é um número livre: ele depende só da
proporção, e vale $\sqrt{p(1-p)}$. Confira:


In [ ]:
p = com_valor["acima_de_10k"].mean()

pd.Series({
    "desvio padrão pelo pandas": com_valor["acima_de_10k"].std(ddof=0),
    "raiz de p(1-p)": (p * (1 - p)) ** 0.5,
}).round(4)


## Cada tipo, sua estatística

Em variável nominal, média não existe. O que existe é a **moda**, que é a
categoria mais frequente, e a proporção de cada categoria:


In [ ]:
com_valor["comarca"].mode()


In [ ]:
com_valor["comarca"].value_counts(normalize=True).head(5).round(3)


Resumindo o que cabe em cada tipo:

| tipo | posição | dispersão | o que **não** fazer |
|---|---|---|---|
| numérica contínua | média, mediana | desvio padrão, IQR | reportar só a média em distribuição assimétrica |
| numérica discreta | média, mediana, moda | desvio padrão, IQR | esquecer que a média pode ser fracionária |
| categórica ordinal | mediana, moda | amplitude de postos | média das categorias |
| categórica nominal | moda | nenhuma clássica | média, mediana, desvio padrão |
| categórica binária | proporção (= média) | $\sqrt{p(1-p)}$ | tratar como contínua |
| identificador | nenhuma | nenhuma | qualquer conta |


## Respondendo à pergunta

Juntando filtro e estatística: o valor típico no Direito Privado, comparado com
o geral. Na aula 4 você vai ver o `groupby`, que compara todos os grupos de uma
vez; por enquanto separamos a base em pedaços.


In [ ]:
privado = com_valor[com_valor["secao"] == "Direito Privado"]

pd.Series({
    "n": len(privado),
    "mediana": privado["valor_indenizacao"].median(),
    "média": privado["valor_indenizacao"].mean(),
    "desvio padrão": privado["valor_indenizacao"].std(),
    "proporção acima de 10 mil": privado["acima_de_10k"].mean(),
}).round(2)


**Agora você.** Monte o mesmo resumo para os acórdãos julgados em 2026. Lembre do `.dt.year` da aula 2.


In [ ]:
em_2026 = com_valor[com_valor["data_julgamento"].dt.________ == 2026]

pd.Series({
    "n": len(em_2026),
    "mediana": em_2026["valor_indenizacao"].________(),
    "média": em_2026["valor_indenizacao"].mean(),
    "desvio padrão": em_2026["valor_indenizacao"].________(),
    "proporção acima de 10 mil": em_2026["acima_de_10k"].mean(),
}).round(2)


### Exercício 2

A base tem 460 acórdãos e 303 com valor extraído. Escreva, em duas ou três
linhas, como você reportaria esse número numa seção de metodologia, e que
problema isso pode causar na interpretação da mediana.


In [ ]:
# exercício 2 (responda em célula de texto)


## O que ficou

1. **Variável e estatística são coisas diferentes.** A variável é a coluna, o
   valor observado é a célula, a estatística é o resumo da coluna inteira.
2. **Filtrar é decisão de pesquisa.** O recorte precisa estar escrito e
   justificado, porque ele define sobre o que a resposta vale.
3. **A estatística tem que caber no tipo.** Mediana em contínua assimétrica,
   proporção em binária, moda em nominal.
4. **Média de binária é proporção**, e desvio padrão amostral divide por $n-1$.

Na aula 4 vamos escrever tudo isso de forma mais curta, encadeando as operações,
e comparar todos os grupos de uma vez com `groupby`. Para praticar filtros antes
disso, use o notebook `extra_filtros_plano_saude.ipynb`, que está completo.
